# **Import dependencies**

In [1]:
from google.colab import drive
import os
import glob
import ee
import geemap
import geopandas as gpd
#import geobr
import json
#import rasterio as rio
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.image as image
import imageio
import ipywidgets as widgets
from ipywidgets import HBox, Label, Layout
#from rasterio.plot import show
from matplotlib.colors import ListedColormap
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
#from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib.offsetbox import OffsetImage
from IPython.display import display, clear_output

In [2]:
import google.colab.userdata
# Desativa a busca por senhas na interface web do Colab para não travar o VS Code
google.colab.userdata.get = lambda key: None

In [3]:
# Solve map not plotted problem
!pip install ipyleaflet>=0.19.2

In [4]:
# Authentication and project information in GEE
ee.Authenticate()
ee.Initialize(project='davi-carbono-ufjf')

In [5]:
# Connect with Google Drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


# **MapBiomas settings**

In [6]:
## Adding MapBiomas GPP data (Carbon Emphasis)
import ee

# 1. Caminho correto do GPP (Coleção 9)
path_project = 'projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_pasture_gpp_v1'

# 2. GPP é uma Image (multibanda), por isso usamos ee.Image direto
mapbiomas = ee.Image(path_project)

# 3. Print das bandas (Anos de 2000 a 2022)
print('✅ Bandas de GPP disponíveis:', mapbiomas.bandNames().getInfo())

# Created based on : https://brasil.mapbiomas.org/wp-content/uploads/sites/4/2023/08/Legenda-Colecao-8-LEGEND-CODE.pdf
dicionario_cores = {
    1: "#32a65e",
    3: "#1f8d49",
    4: "#7dc975",
    5: "#04381d",
    6: "#026975",
    49: "#02d659",
    10: "#ad975a",
    11: "#519799",
    12: "#d6bc74",
    32: "#fc8114",
    29: "#ffaa5f",
    50: "#ad5100",
    13: "#d89f5c",
    14: "#FFFFB2",
    15: "#edde8e",
    18: "#E974ED",
    19: "#C27BA0",
    39: "#f5b3c8",
    20: "#db7093",
    40: "#c71585",
    62: "#ff69b4",
    41: "#f54ca9",
    36: "#d082de",
    46: "#d68fe2",
    47: "#9932cc",
    48: "#e6ccff",
    9: "#7a5900",
    21: "#ffefc3",
    22: "#d4271e",
    23: "#ffa07a",
    24: "#d4271e",
    30: "#9c0027",
    25: "#db4d4f",
    26: "#0000FF",
    33: "#2532e4",
    31: "#091077",
    27: "#ffffff"
}

dicionario_classes = {
    1: "Floresta",
    3: "Formação Florestal",
    4: "Formação Savânica",
    5: "Mangue",
    6: "Floresta Alagável (beta)",
    49: "Restinga Arborizada",
    10: "Formação Natural não Florestal",
    11: "Campo Alagado e Área Pantanosa",
    12: "Formação Campestre",
    32: "Apicum",
    29: "Afloramento Rochoso",
    50: "Restinga Herbácea",
    13: "Outras Formações não Florestais",
    14: "Agropecuária",
    15: "Pastagem",
    18: "Agricultura",
    19: "Lavoura Temporária",
    39: "Soja",
    20: "Cana",
    40: "Arroz",
    62: "Algodão (beta)",
    41: "Outras Lavouras Temporárias",
    36: "Lavoura Perene",
    46: "Café",
    47: "Citrus",
    48: "Outras Lavouras Perenes",
    9: "Silvicultura",
    21: "Mosaico de Usos",
    22: "Área não Vegetada",
    23: "Praia, Duna e Areal",
    24: "Área Urbanizada",
    30: "Mineração",
    25: "Outras Áreas não Vegetadas",
    26: "Corpo D'água",
    33: "Rio, Lago e Oceano",
    31: "Aquicultura",
    27: "Não observado"
}

## Pallete
paleta_nomes = {key:value for key, value in zip(dicionario_classes.values(), dicionario_cores.values())}
paleta_nomes

# Creating the color palette
paleta_cores = {
    0: "#ffffff",1: "#32a65e", 2: "#32a65e",3: "#1f8d49",
    4: "#7dc975",5: "#04381d", 6: "#026975",7: "#000000",
    8: "#000000",9: "#7a6c00", 10: "#ad975a",11: "#519799",
    12: "#d6bc74",13: "#d89f5c", 14: "#FFFFB2",15: "#edde8e",
    16: "#000000",17: "#000000", 18: "#f5b3c8",19: "#C27BA0",
    20: "#db7093",21: "#ffefc3", 22: "#db4d4f",23: "#ffa07a",
    24: "#d4271e",25: "#db4d4f", 26: "#0000FF",27: "#000000",
    28: "#000000",29: "#ffaa5f", 30: "#9c0027",31: "#091077",
    32: "#fc8114",33: "#2532e4", 34: "#93dfe6",35: "#9065d0",
    36: "#d082de",37: "#000000", 38: "#000000",39: "#f5b3c8",
    40: "#c71585",41: "#f54ca9", 42: "#cca0d4",43: "#dbd26b",
    44: "#807a40",45: "#e04cfa", 46: "#d68fe2",47: "#9932cc",
    48: "#e6ccff",49: "#02d659", 50: "#ad5100",51: "#000000",
    52: "#000000",53: "#000000", 54: "#000000",55: "#000000",
    56: "#000000",57: "#CC66FF", 58: "#FF6666",59: "#006400",
    60: "#8d9e8b",61: "#f5d5d5", 62: "#ff69b4"
}
palette_list = list(paleta_cores.values())
len(palette_list)
vis_gpp = {
    'min': 0,
    'max': 3000, # Valor comum de gC/m2/ano em pastagens produtivas
    'palette': ['#f7fcb9', '#addd8e', '#31a354'] # Do amarelo claro ao verde escuro
}

# 4. Exemplo de como selecionar e visualizar um ano específico (2021)
gpp_2021 = mapbiomas.select('gpp_2021')

✅ Bandas de GPP disponíveis: ['gpp_2000', 'gpp_2001', 'gpp_2002', 'gpp_2003', 'gpp_2004', 'gpp_2005', 'gpp_2006', 'gpp_2007', 'gpp_2008', 'gpp_2009', 'gpp_2010', 'gpp_2011', 'gpp_2012', 'gpp_2013', 'gpp_2014', 'gpp_2015', 'gpp_2016', 'gpp_2017', 'gpp_2018', 'gpp_2019', 'gpp_2020', 'gpp_2021', 'gpp_2022']


# **Defining and plot the map**



In [7]:
# 1. Definindo o Estado (Voltando para 2021, que é o que o Luiz tem)
# Se você quiser Minas Gerais, mantenha o filtro '31'. Se for Rondônia, mude para '11'.
state = ee.FeatureCollection('projects/ee-lzfsantos/assets/MG_UF_2021').filter(ee.Filter.eq('CD_UF','31'))

# 2. Definindo a Cidade (Pedra Dourada - Código 3110103)
city = ee.FeatureCollection('projects/ee-lzfsantos/assets/MG_Municipios_2021').filter(ee.Filter.eq('CD_MUN','3110103'))

# 3. Definindo a Fazenda (Usando o ID que você filtrou)
farm = ee.FeatureCollection('projects/ee-lzfsantos/assets/rg_se_landtenure_imaflora_201810_sirgas').filter(ee.Filter.eq('id_imovel', 3851328))

# 4. Camada de Carbono (Mudança de 'classification' para 'gpp')
# Como você disse que quer foco em Carbono, usamos a banda de GPP
map_carbon_2021 = mapbiomas.select('gpp_2021')

# 5. Recorte e Visualização
mapbiomas_clip = map_carbon_2021.clip(farm)

Map = geemap.Map()
Map.addLayer(state, {'color': 'gray'}, "State Border")
Map.addLayer(city, {'color': 'blue'}, "City Border")

# Para o GPP, a paleta é diferente (geralmente do vermelho ao verde escuro)
Map.addLayer(mapbiomas_clip, {'min': 0, 'max': 3000, 'palette': ['red', 'yellow', 'green']}, 'Carbon Sequestration (GPP 2021)')

Map.centerObject(farm, 14) # Aumentei o zoom para 14 para ver melhor a fazenda
Map

Map(center=[-20.72614763169431, -41.902187629793715], controls=(WidgetControl(options=['position', 'transparen…

In [8]:
# Teste rápido: Qual a classe de uso do solo dessa fazenda?
# Usando a classificação geral (que você já acessou antes)
lulc_2021 = ee.Image('projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_integration_v1').select('classification_2021')

# Pega o valor mais frequente dentro da fazenda
classe_frequente = lulc_2021.reduceRegion(
    reducer=ee.Reducer.mode(),
    geometry=farm.geometry(),
    scale=30
).get('classification_2021').getInfo()

print(f"ID da Classe MapBiomas na fazenda: {classe_frequente}")
# Se der 15, é pasto. Se der 3, é floresta. Se der 39/46/48, é agricultura.

ID da Classe MapBiomas na fazenda: 15


In [9]:
import pandas as pd
import ipywidgets as widgets
import ee
import geemap
from IPython.display import display, clear_output

# 1. Carregando o arquivo atualizado
all_farms_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/files/output/final/data_integration_all_areas_final.csv")

# 2. Recipiente para o mapa
output_mapa = widgets.Output()

# --- Configuração dos Widgets ---
list_state = sorted(list(all_farms_df['state_name'].unique()))
state_name = widgets.Dropdown(options=list_state, description='Estado:', value=list_state[0] if list_state else None)

def get_cities(state):
    return sorted(list(all_farms_df[all_farms_df['state_name'] == state]['city_name'].unique()))

def get_farms(city):
    return sorted(list(all_farms_df[all_farms_df['city_name'] == city]['farm'].unique()))

city_name = widgets.Dropdown(options=get_cities(state_name.value), description='Cidade:')
farm_name = widgets.Dropdown(options=get_farms(city_name.value), description='Fazenda:')

# Campos de Resultado (Nomes atualizados conforme o CSV)
farm_area = widgets.FloatText(description='Área (ha):', disabled=True)
biome_name = widgets.Text(description='Bioma:', disabled=True)
climate_name = widgets.Text(description='Clima:', disabled=True)
balance_farm = widgets.FloatText(description='CO2 Total:', disabled=True)
balance_farm_ha = widgets.FloatText(description='tCO2/ha:', disabled=True)
year_label = widgets.Label(value="📅 Dados de 2021 (Foco em Carbono/GPP)")

# --- Funções de Atualização ---

def update_map(change):
    with output_mapa:
        clear_output(wait=True)

        try:
            farm_id = farm_name.value
            # Filtra a linha da fazenda selecionada
            data = all_farms_df[all_farms_df['farm'] == farm_id].iloc[0]

            # Atualiza os widgets com os nomes de colunas corretos do seu CSV
            farm_area.value = data['area_size']
            biome_name.value = data['biome']
            climate_name.value = data['climate']
            balance_farm.value = data['balance_CO2_area']
            balance_farm_ha.value = data['balance_CO2_ha']

            global city_cod, farm_cod
            city_cod = data['city_cod']
            farm_cod = farm_id

            # --- Processamento GEE ---
            # Define o Asset regional do Luiz
            if state_name.value == 'Rondônia':
                asset = 'projects/ee-lzfsantos/assets/rg_n_landtenure_imaflora_201810_sirgas'
            elif state_name.value in ['Minas Gerais', 'Ceará']:
                asset = 'projects/ee-lzfsantos/assets/rg_se_landtenure_imaflora_201810_sirgas'
            elif state_name.value == 'Rio Grande do Sul':
                asset = 'projects/ee-lzfsantos/assets/rg_s_landtenure_imaflora_201810_sirgas'
            else:
                asset = 'projects/ee-lzfsantos/assets/rg_co_landtenure_imaflora_201810_sirgas'

            farm_feature = ee.FeatureCollection(asset).filter(ee.Filter.eq('id_imovel', int(farm_id)))

            # Camada de Carbono (GPP 2021)
            # Certifique-se que executou a célula do 'mapbiomas = ee.Image(...gpp_v1)' antes!
            map_carbon = mapbiomas.select('gpp_2021')
            clip_carbon = map_carbon.clip(farm_feature)

            # Visualização
            M = geemap.Map()
            M.addLayer(clip_carbon, {'min': 0, 'max': 3000, 'palette': ['#f7fcb9', '#addd8e', '#31a354']}, 'Carbono (GPP)')
            M.centerObject(farm_feature, 13)
            display(M)
            print(f"✅ Fazenda {farm_id} carregada com sucesso.")

        except Exception as e:
            print(f"⚠️ Erro ao carregar dados: {e}")

# --- Observers (Eventos) ---

def on_state_change(change):
    city_name.options = get_cities(change.new)
state_name.observe(on_state_change, names='value')

def on_city_change(change):
    farm_name.options = get_farms(change.new)
city_name.observe(on_city_change, names='value')

farm_name.observe(update_map, names='value')

# --- Layout Final ---
col1 = widgets.VBox([state_name, city_name, farm_name])
col2 = widgets.VBox([farm_area, biome_name, climate_name])
col3 = widgets.VBox([balance_farm, balance_farm_ha])
ui = widgets.HBox([col1, col2, col3])

display(ui, year_label, output_mapa)

# Carga inicial
update_map(None)

Label(value='📅 Dados de 2021 (Foco em Carbono/GPP)')

Output()

In [10]:
Map

Map(center=[-20.72614763169431, -41.902187629793715], controls=(WidgetControl(options=['position', 'transparen…

In [11]:
def update_map(change):
    with output_mapa:
        clear_output(wait=True)

        try:
            farm_id = farm_name.value
            data = all_farms_df[all_farms_df['farm'] == farm_id].iloc[0]

            # Atualiza os textos da interface
            farm_area.value = data['area_size']
            biome_name.value = data['biome']
            climate_name.value = data['climate']
            balance_farm.value = data['balance_CO2_area']
            balance_farm_ha.value = data['balance_CO2_ha']

            # --- Lógica GEE ---
            if state_name.value == 'Rondônia':
                asset = 'projects/ee-lzfsantos/assets/rg_n_landtenure_imaflora_201810_sirgas'
            elif state_name.value in ['Minas Gerais', 'Ceará']:
                asset = 'projects/ee-lzfsantos/assets/rg_se_landtenure_imaflora_201810_sirgas'
            elif state_name.value == 'Rio Grande do Sul':
                asset = 'projects/ee-lzfsantos/assets/rg_s_landtenure_imaflora_201810_sirgas'
            else:
                asset = 'projects/ee-lzfsantos/assets/rg_co_landtenure_imaflora_201810_sirgas'

            # Criando a farm_feature (agora ela existe aqui dentro!)
            farm_feature = ee.FeatureCollection(asset).filter(ee.Filter.eq('id_imovel', int(farm_id)))

            # Camada de Carbono
            map_carbon = mapbiomas.select('gpp_2021')
            mapbiomas_clip = map_carbon.clip(farm_feature)

            # --- Configuração do Mapa com Legendas ---
            M = geemap.Map()

            # 1. Adicionando a Colorbar (Termômetro de Carbono)
            vis_params = {'min': 0, 'max': 3000, 'palette': ['#f7fcb9', '#addd8e', '#31a354']}
            M.add_colorbar(
                vis_params=vis_params,
                label="Sequestro de Carbono (gC/m²/ano)",
                layer_name="Carbono (GPP)"
            )

            # 2. Adicionando Legenda Manual de Classes (Simples)
            legend_dict = {
                'Pastagem': '#edde8e',
                'Floresta': '#1f8d49',
                'Agricultura': '#E974ED'
            }
            M.add_legend(title="Uso do Solo", legend_dict=legend_dict, position='bottomleft')

            # 3. Adicionando as camadas
            M.addLayer(mapbiomas_clip, vis_params, 'Carbono (GPP)')
            M.centerObject(farm_feature, 13)

            display(M)
            print(f"✅ Fazenda {farm_id} carregada com legendas.")

        except Exception as e:
            print(f"⚠️ Erro ao carregar dados: {e}")

# Reatribuindo o evento para garantir que use a função nova
farm_name.observe(update_map, names='value')
display(ui, year_label, output_mapa)
update_map(None)

Label(value='📅 Dados de 2021 (Foco em Carbono/GPP)')

Output()

In [12]:
display(ui)

In [14]:
import ipywidgets as widgets
from IPython.display import display

botao = widgets.Button(description="Teste VS Code")
display(botao)

Button(description='Teste VS Code', style=ButtonStyle())

In [13]:
# 1. Definindo o Estado (Usando a base GAUL - Pública e Estável)
# MG = 31, RO = 11, etc.
match state_name.value:
    case 'Ceará': adm1_name = 'Ceara'
    case 'Goiás': adm1_name = 'Goias'
    case 'Mato Grosso': adm1_name = 'Mato Grosso'
    case 'Mato Grosso do Sul': adm1_name = 'Mato Grosso Do Sul'
    case 'Minas Gerais': adm1_name = 'Minas Gerais'
    case 'Rio Grande do Sul': adm1_name = 'Rio Grande Do Sul'
    case 'Rondônia': adm1_name = 'Rondonia'
    case _: adm1_name = 'Minas Gerais'

# Carrega os estados do mundo e filtra pelo Brasil e pelo nome do seu estado
state = ee.FeatureCollection("FAO/GAUL/2015/level1") \
          .filter(ee.Filter.eq('ADM0_NAME', 'Brazil')) \
          .filter(ee.Filter.eq('ADM1_NAME', adm1_name))

# 2. Definindo a Fazenda (Usando os caminhos regionais do Luiz Fernando)
if state_name.value in ['Ceará', 'Minas Gerais']:
    asset_path = 'projects/ee-lzfsantos/assets/rg_se_landtenure_imaflora_201810_sirgas'
elif state_name.value in ['Goiás', 'Mato Grosso', 'Mato Grosso do Sul']:
    asset_path = 'projects/ee-lzfsantos/assets/rg_co_landtenure_imaflora_201810_sirgas'
elif state_name.value == 'Rio Grande do Sul':
    asset_path = 'projects/ee-lzfsantos/assets/rg_s_landtenure_imaflora_201810_sirgas'
else: # Rondônia
    asset_path = 'projects/ee-lzfsantos/assets/rg_n_landtenure_imaflora_201810_sirgas'

# Usamos o ID que vem dos seus widgets
farm = ee.FeatureCollection(asset_path).filter(ee.Filter.eq('id_imovel', int(farm_cod)))

# 3. Camada de Carbono (MUDANÇA CRUCIAL: gpp_2021)
# O seu código antigo tentava selecionar 'classification_2021', que não existe no arquivo de GPP
map_carbon = mapbiomas.select('gpp_2021')
mapbiomas_clip = map_carbon.clip(farm)

# 4. Plotando o Mapa
Map = geemap.Map()

# Adiciona o limite do Estado (Cinza)
Map.addLayer(state, {'color': 'gray'}, "Limite Estadual (GAUL)")

# Adiciona o Carbono (GPP) com a paleta de créditos de carbono
vis_gpp = {'min': 0, 'max': 3000, 'palette': ['#f7fcb9', '#addd8e', '#31a354']}
Map.addLayer(mapbiomas_clip, vis_gpp, 'Sequestro de Carbono (GPP 2021)')

# Adiciona a Colorbar para o seu relatório
Map.add_colorbar(vis_params=vis_gpp, label="Sequestro de Carbono (gC/m²/ano)")

Map.centerObject(farm, 12)
Map

NameError: name 'farm_cod' is not defined